# Data preprocessing

Evaluate duplicate records and apply the approved missing-value treatment.
Exploratory findings are documented in `data_exploration.ipynb`.

Run this notebook from the `notebooks/` directory with Python and pandas installed. The input is `../data/raw/student_data.csv`. Each notebook loads the input independently. The previous user environment used Python 3.14.10 and pandas 3.1.1. Execution outputs below record the validation runtime.

### Environment check

The Python interpreter, version, working directory, and pandas version
are displayed to record the execution environment.

In [1]:
import sys
import os
import pandas as pd
from pathlib import Path

print(sys.executable)
print(sys.version)
print(os.getcwd())
print(pd.__version__)

/opt/homebrew/opt/python@3.14/bin/python3.14
3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 17.0.0 (clang-1700.6.3.2)]
/Users/yassir/Data-Mining-Project/notebooks
3.0.6


### Data loading and initial dimensions

The CSV is loaded into the DataFrame `data`.
The loading options preserve the literal education category "None"
while treating empty fields as missing values.

The imported dataset contains 649 rows and 34 columns.
One column is an exported row index, so these are not 34 analytical features.

In [2]:
data = pd.read_csv(
    '../data/raw/student_data.csv',
    keep_default_na=False,
    na_values=['']
)

data.shape

(649, 34)

### Student variables

The exported row index is excluded to create the table of student
variables used for preprocessing.

In [3]:
student_data = data.drop(columns=['Unnamed: 0'])

student_data.shape

(649, 33)

### Missing values before preprocessing

Missing counts and percentages are recorded before applying the selected treatment.

In [4]:
missing_counts = data.isna().sum()
missing_percentages = (missing_counts / len(data) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing count': missing_counts,
    'Missing (%)': missing_percentages
})

missing_summary

,Missing count,Missing (%)
Unnamed: 0,0,0.00
school,1,0.15
sex,10,1.54
age,0,0.00
address,17,2.62
famsize,3,0.46
Pstatus,28,4.31
Medu,27,4.16
Fedu,19,2.93
Mjob,6,0.92


### Treatment of missing absences and failures

A working copy of the dataset is created for preprocessing.
Missing absences and previous failures are replaced with zero,
assuming that blanks represent no absences or failures.
This interpretation remains subject to source confirmation.

In [5]:
student_data_clean = student_data.copy()

student_data_clean['absences'] = student_data_clean['absences'].fillna(0)
student_data_clean['failures'] = student_data_clean['failures'].fillna(0)

### Validation of zero replacement

Before-and-after counts verify the replacement of missing absences and failures with zero.

In [6]:
replacement_check = pd.DataFrame({
    'Missing before': student_data[['absences', 'failures']].isna().sum(),
    'Missing after': student_data_clean[['absences', 'failures']].isna().sum(),
    'Zeros after': student_data_clean[['absences', 'failures']].eq(0).sum()
})

replacement_check

,Missing before,Missing after,Zeros after
absences,260,0,260
failures,551,0,551


### Row preservation and remaining missing values

Row counts and order are checked after filling. Remaining missing values
are displayed for later preprocessing decisions.

In [7]:
assert student_data_clean.shape == student_data.shape
assert student_data_clean.index.equals(student_data.index)
assert student_data_clean[['absences', 'failures']].isna().sum().sum() == 0

student_data_clean.isna().sum()

school         1
sex           10
age            0
address       17
famsize        3
Pstatus       28
Medu          27
Fedu          19
Mjob           6
Fjob          18
reason         7
guardian       8
traveltime    11
studytime     14
failures       0
schoolsup     30
famsup         7
paid          31
activities    31
nursery       20
higher        21
internet       5
romantic       1
famrel        24
freetime      20
goout         31
Dalc          24
Walc          27
health        17
absences       0
G1             0
G2             0
G3             0
dtype: int64

### Order of remaining treatments

Remaining missing values are sorted from most to least frequent.
Variables are treated in this order, and variables of the same type
share one treatment.

In [8]:
remaining_missing = student_data_clean.isna().sum()

remaining_missing = remaining_missing[remaining_missing > 0].sort_values(
    ascending=False
)

remaining_missing

paid          31
goout         31
activities    31
schoolsup     30
Pstatus       28
Walc          27
Medu          27
Dalc          24
famrel        24
higher        21
freetime      20
nursery       20
Fedu          19
Fjob          18
health        17
address       17
studytime     14
traveltime    11
sex           10
guardian       8
famsup         7
reason         7
Mjob           6
internet       5
famsize        3
romantic       1
school         1
dtype: int64

### Treatment of missing Yes/No responses

The eight Yes/No features contain between 1 (`romantic`) and 31
(`paid`, `activities`) missing responses. Missing responses are recorded
as a separate "Unknown" category, so no answer is guessed for a student.

`data_exploration.ipynb` (Missing-value overlap in Yes/No features) shows
that missing responses rarely occur together. Of the 140 students missing
at least one Yes/No response, 135 are missing exactly one, and the largest
pairwise overlap involves two students. "Unknown" therefore marks
individual unanswered questions rather than a group of students who left
the questionnaire incomplete.

"Unknown" adds a third level to each Yes/No feature. This must be
considered when the features are encoded.

In [9]:
yes_no_columns = [
    'schoolsup', 'famsup', 'paid', 'activities',
    'nursery', 'higher', 'internet', 'romantic'
]

for column in yes_no_columns:
    student_data_clean[column] = student_data_clean[column].fillna('Unknown')

### Validation of Yes/No replacement

Before-and-after counts verify that every missing response became
"Unknown" and that recorded "yes" and "no" answers are unchanged.

In [10]:
yes_no_check = pd.DataFrame({
    'Missing before': student_data[yes_no_columns].isna().sum(),
    'Missing after': student_data_clean[yes_no_columns].isna().sum(),
    'Unknown after': student_data_clean[yes_no_columns].eq('Unknown').sum()
})

assert student_data_clean[yes_no_columns].eq('yes').sum().equals(
    student_data[yes_no_columns].eq('yes').sum()
)
assert student_data_clean[yes_no_columns].eq('no').sum().equals(
    student_data[yes_no_columns].eq('no').sum()
)

yes_no_check

,Missing before,Missing after,Unknown after
schoolsup,30,0,30
famsup,7,0,7
paid,31,0,31
activities,31,0,31
nursery,20,0,20
higher,21,0,21
internet,5,0,5
romantic,1,0,1


### Study time by school

Missing study times are counted by school, and the median study time of
each school is calculated from recorded values. Every student with a
missing study time has a recorded school.

In [11]:
study_by_school = pd.DataFrame({
    'Missing studytime': student_data_clean.loc[
        student_data_clean['studytime'].isna(), 'school'
    ].value_counts(),
    'Median studytime': student_data_clean.groupby('school')['studytime'].median()
})

study_by_school

,Missing studytime,Median studytime
school,,
GP,8,203.5
MS,6,156.0


### Treatment of missing study time

GP students report more weekly study time than MS students, so a single
overall median would understate study time at GP and overstate it at MS.
Each missing value is replaced with the median of the student's school.
The median is used because study time is strongly right-skewed.

`transform('median')` returns, for every student, the median of their
school, aligned with the original rows. `fillna` uses that value only
where study time is missing.

In [12]:
school_study_median = student_data_clean.groupby('school')['studytime'].transform('median')

student_data_clean['studytime'] = student_data_clean['studytime'].fillna(school_study_median)

### Validation of study-time replacement

The filled values are displayed by school. The checks confirm that no
study time is missing and that recorded values are unchanged.

In [13]:
filled_study = student_data_clean.loc[
    student_data['studytime'].isna(),
    ['school', 'studytime']
]

assert student_data_clean['studytime'].isna().sum() == 0
assert student_data_clean['studytime'].where(
    student_data['studytime'].notna()
).equals(student_data['studytime'])

filled_study.value_counts()

school  studytime
GP      203.5        8
MS      156.0        6
Name: count, dtype: int64

### Travel time by school and residence

Missing travel times are counted by school and residence type, and the
median travel time of each combination is calculated from recorded values.
Every student with a missing travel time has a recorded school and
residence. `data_exploration.ipynb` (Travel time, residence, and school)
shows that rural students travel longer than urban students within both
schools.

In [14]:
travel_by_group = pd.DataFrame({
    'Missing traveltime': student_data_clean.loc[
        student_data_clean['traveltime'].isna(), ['school', 'address']
    ].value_counts(),
    'Median traveltime': student_data_clean.groupby(
        ['school', 'address']
    )['traveltime'].median()
})

travel_by_group['Missing traveltime'] = (
    travel_by_group['Missing traveltime'].fillna(0).astype(int)
)

travel_by_group

Missing traveltime  Median traveltime
school address                                       
GP     R                         1               19.0
       U                         7                9.0
MS     R                         3               25.0
       U                         0               17.0

### Treatment of missing travel time

Travel time depends on the school and on where the student lives, so a
single overall median would understate rural journeys and overstate urban
ones. Each missing value is replaced with the median of students from the
same school and residence type.

In [15]:
group_travel_median = student_data_clean.groupby(
    ['school', 'address']
)['traveltime'].transform('median')

student_data_clean['traveltime'] = student_data_clean['traveltime'].fillna(group_travel_median)

### Validation of travel-time replacement

The filled values are displayed by school and residence. The checks
confirm that no travel time is missing and that recorded values are
unchanged.

In [16]:
filled_travel = student_data_clean.loc[
    student_data['traveltime'].isna(),
    ['school', 'address', 'traveltime']
]

assert student_data_clean['traveltime'].isna().sum() == 0
assert student_data_clean['traveltime'].where(
    student_data['traveltime'].notna()
).equals(student_data['traveltime'])

filled_travel.value_counts()

school  address  traveltime
GP      U        9.0           7
MS      R        25.0          3
GP      R        19.0          1
Name: count, dtype: int64

### Rating medians

The median of each 1–5 rating feature is calculated from recorded
responses before filling.

In [17]:
rating_columns = [
    'famrel', 'freetime', 'goout',
    'Dalc', 'Walc', 'health'
]

rating_medians = student_data_clean[rating_columns].median()

rating_medians

famrel      4.0
freetime    3.0
goout       3.0
Dalc        1.0
Walc        2.0
health      4.0
dtype: float64

### Treatment of missing ratings

The six rating features contain between 17 (`health`) and 31 (`goout`)
missing responses. Each missing response is replaced with the median of
its feature. The median of an ordered 1–5 scale is always a valid
response level and is not affected by extreme responses.

`data_exploration.ipynb` (Missing-value overlap in rating features) shows
that missing ratings rarely occur together. Of the 129 students missing
at least one rating, 115 are missing exactly one, and the largest
pairwise overlap involves three students. Each feature can therefore be
filled separately.

In [18]:
for column in rating_columns:
    student_data_clean[column] = student_data_clean[column].fillna(
        rating_medians[column]
    )

### Validation of rating replacement

Before-and-after counts verify that no rating is missing, that every
filled value equals the feature median, and that recorded responses are
unchanged.

In [19]:
rating_check = pd.DataFrame({
    'Missing before': student_data[rating_columns].isna().sum(),
    'Missing after': student_data_clean[rating_columns].isna().sum(),
    'Median used': rating_medians
})

assert student_data_clean[rating_columns].isna().sum().sum() == 0
assert student_data_clean[rating_columns].where(
    student_data[rating_columns].notna()
).equals(student_data[rating_columns])

rating_check

,Missing before,Missing after,Median used
famrel,24,0,4.0
freetime,20,0,3.0
goout,31,0,3.0
Dalc,24,0,1.0
Walc,27,0,2.0
health,17,0,4.0


### Treatment of missing categorical values

Missing parental status, parental education, parental occupation,
reason for choosing the school, and residence type are recorded as a
separate unknown category rather than guessed. The observed values give
no basis for inferring, for example, a parent's occupation or education.

- `Pstatus` uses "U", following the single-letter coding of "T"
  (together) and "A" (apart).
- The other features use "Unknown". In `address`, "U" already means
  urban, so a single letter would merge unknown residences with urban
  ones.
- In `Medu` and `Fedu`, "Unknown" remains distinct from the recorded
  level "None" (no formal education).

The unknown category has no position on the education scale. This must
be considered when parental education is encoded.

In [20]:
student_data_clean['Pstatus'] = student_data_clean['Pstatus'].fillna('U')

unknown_columns = ['Medu', 'Fedu', 'Fjob', 'address', 'reason', 'Mjob']

for column in unknown_columns:
    student_data_clean[column] = student_data_clean[column].fillna('unknown')

### Validation of categorical replacement

Before-and-after counts verify that every missing value became the
unknown category and that recorded categories are unchanged.

In [21]:
category_columns = ['Pstatus'] + unknown_columns

unknown_after = student_data_clean[unknown_columns].eq('unknown').sum()
unknown_after['Pstatus'] = student_data_clean['Pstatus'].eq('U').sum()

category_check = pd.DataFrame({
    'Missing before': student_data[category_columns].isna().sum(),
    'Missing after': student_data_clean[category_columns].isna().sum(),
    'Unknown after': unknown_after
}).loc[category_columns]

assert student_data_clean[category_columns].isna().sum().sum() == 0
assert student_data_clean[category_columns].where(
    student_data[category_columns].notna()
).equals(student_data[category_columns])

category_check

,Missing before,Missing after,Unknown after
Pstatus,28,0,28
Medu,27,0,27
Fedu,19,0,19
Fjob,18,0,18
address,17,0,17
reason,7,0,7
Mjob,6,0,6


### Treatment of missing sex

Ten students have no recorded sex. The missing values are recorded as a
separate category, "U", rather than assigned to the more frequent
category. Following the single-letter coding of "F" and "M", "U" marks
an unknown value, as in `Pstatus`.

In [22]:
student_data_clean['sex'] = student_data_clean['sex'].fillna('U')

### Validation of sex replacement

Category counts before and after filling are compared.

In [23]:
sex_check = pd.DataFrame({
    'Before': student_data['sex'].value_counts(dropna=False),
    'After': student_data_clean['sex'].value_counts(dropna=False)
}).fillna(0).astype(int)

assert student_data_clean['sex'].isna().sum() == 0

sex_check

,Before,After
sex,,
F,377,377
M,262,262
U,0,10
NaN,10,0


### Treatment of missing guardian

Eight students have no recorded guardian. The missing values are replaced
with the most frequent guardian (the mode). Mothers are the recorded
guardian for most students, and eight values are too few to change the
distribution meaningfully.

In [24]:
guardian_mode = student_data_clean['guardian'].mode()[0]

student_data_clean['guardian'] = student_data_clean['guardian'].fillna(guardian_mode)

### Validation of guardian replacement

Guardian counts before and after filling are compared.

In [25]:
guardian_check = pd.DataFrame({
    'Before': student_data['guardian'].value_counts(dropna=False),
    'After': student_data_clean['guardian'].value_counts(dropna=False)
}).fillna(0).astype(int)

assert student_data_clean['guardian'].isna().sum() == 0

guardian_check

,Before,After
guardian,,
father,150,150
mother,450,458
other,41,41
NaN,8,0


### Family size by parental status

`famsize` counts the people in the student's family, including the student.
A plausible family size is therefore at least 2: the student and at least
one guardian. Recorded sizes of 0 and 1 are treated in a later step and are
left out of all calculations below.

To replace missing and implausible values with a value based on similar
students rather than one value for everyone, family size was compared across
the other variables, using plausible family sizes only. Two variables show
a difference with a clear explanation: parental status (this section) and
guardian (next section). Medians for school, address, sex, parental
education, family support, and internet access are the same in every group,
and differences by parental occupation and reason for choosing the school
are small and have no clear explanation.

Among students with a plausible family size, families whose parents live
apart are about one person smaller, which is expected when one parent lives
elsewhere. The table counts the missing family sizes and gives the median
plausible family size for each parental status.

In [26]:
plausible_family = student_data['famsize'] >= 2

family_by_status = pd.DataFrame({
    'Missing famsize': student_data.loc[
        student_data['famsize'].isna(), 'Pstatus'
    ].value_counts(),
    'Plausible famsize records': student_data.loc[
        plausible_family, 'Pstatus'
    ].value_counts(),
    'Median famsize': student_data.loc[plausible_family].groupby(
        'Pstatus'
    )['famsize'].median()
})

family_by_status['Missing famsize'] = (
    family_by_status['Missing famsize'].fillna(0).astype(int)
)

family_by_status

,Missing famsize,Plausible famsize records,Median famsize
Pstatus,,,
A,1,48,5.0
T,2,470,6.0


### Family size by guardian

Family size is compared across guardians, again using plausible family sizes
only. The share of families with seven or more people shows how often each
group has a large family.

In [27]:
plausible_data = student_data.loc[plausible_family]

family_by_guardian = plausible_data.groupby('guardian')['famsize'].agg(
    ['count', 'median', 'mean']
)

family_by_guardian['Families of 7 or more (%)'] = (
    plausible_data['famsize'].ge(7).groupby(plausible_data['guardian']).mean() * 100
)

family_by_guardian.round(2)

,count,median,mean,Families of 7 or more (%)
guardian,,,,
father,130,6.0,5.86,43.08
mother,372,6.0,5.79,40.32
other,34,7.0,6.56,55.88


### Guardian within each parental status

To check that the guardian pattern is not simply a parental-status pattern,
students whose guardian is "other" are compared with students whose guardian
is a parent within each parental status.

In [28]:
guardian_type = plausible_data['guardian'].map({
    'mother': 'mother or father',
    'father': 'mother or father',
    'other': 'other'
})

family_by_status_guardian = plausible_data.groupby(
    ['Pstatus', guardian_type]
)['famsize'].agg(['count', 'median', 'mean'])

family_by_status_guardian.round(2)

count  median  mean
Pstatus guardian                             
A       mother or father     43     5.0  4.77
        other                 5     7.0  6.00
T       mother or father    438     6.0  5.86
        other                27     7.0  6.59

### Interpretation of the guardian pattern

Students whose guardian is neither parent ("other", for example a
grandparent or another relative) come from larger families. Their median
family size is 7, compared with 6 for students whose guardian is the mother
or the father, and their mean is 6.56 compared with 5.79 and 5.86. More than
half of them (55.88%) live in families of seven or more, compared with about
40% of the other students.

The pattern holds within both parental-status groups. Among students whose
parents live together, "other" guardians have a mean family size of 6.59,
compared with 5.86 for parent guardians. Among students whose parents live
apart, the means are 6.00 and 4.77, although only five students in that
group have an "other" guardian.

A plausible explanation is that a student who lives with relatives other
than the parents often belongs to an extended household, such as one that
includes grandparents. Mother and father guardians, in contrast, have almost
identical family sizes, so the guardian adds information only when the
guardian is "other".

The "other" group contains 34 students, and their family sizes range from 3
to 9. Its median is therefore less precise than the parental-status medians,
which rest on 48 and 470 students. To avoid basing a value on very small
groups, the "other" guardian median is calculated from all 34 students
together rather than separately within each parental status.

**Replacement rule used for missing and implausible family sizes:**

1. Guardian recorded as "other": median plausible family size of all
   students with an "other" guardian (7).
2. Otherwise: median plausible family size of students with the same
   parental status (5 for parents apart, 6 for parents together).
3. No recorded parental status: median of all plausible family sizes (6).

Combining every parental status with every guardian was also considered and
not adopted. Mother and father guardians differ by less than 0.2 people on
average, so splitting them would change values because of rounding rather
than real differences, and several combinations contain only four or five
students.

### Treatment of missing family size

Each missing family size is replaced using the rule above. The reference
value is the median plausible family size of students with an "other"
guardian when the guardian is "other", and otherwise the median plausible
family size of students with the same parental status.

The groups use the recorded parental status and guardian from
`student_data`, because missing values in those variables were filled
earlier in this notebook and filled values should not serve as evidence.
`where(plausible_family)` hides the sizes of 0 and 1 so that they do not
affect the medians.

All three students with a missing family size have a recorded parental
status and a mother or father as guardian, so their values come from their
parental-status groups.

In [29]:
status_family_median = (
    student_data['famsize']
    .where(plausible_family)
    .groupby(student_data['Pstatus'])
    .transform('median')
)

other_guardian = student_data['guardian'] == 'other'

other_guardian_median = student_data.loc[
    plausible_family & other_guardian, 'famsize'
].median()

family_reference = status_family_median.mask(other_guardian, other_guardian_median)

student_data_clean['famsize'] = student_data_clean['famsize'].fillna(
    family_reference
)

### Validation of family-size replacement

The filled values are displayed with each student's parental status and
guardian. The checks confirm that no family size is missing and that
recorded values, including the sizes of 0 and 1 still to be treated, are
unchanged.

In [30]:
filled_family = pd.concat([
    student_data.loc[student_data['famsize'].isna(), ['Pstatus', 'guardian']],
    student_data_clean.loc[student_data['famsize'].isna(), 'famsize']
], axis=1)

assert student_data_clean['famsize'].isna().sum() == 0
assert student_data_clean['famsize'].where(
    student_data['famsize'].notna()
).equals(student_data['famsize'])

filled_family

,Pstatus,guardian,famsize
220,A,mother,5.0
260,T,mother,6.0
446,T,father,6.0


### Implausible family sizes

Because `famsize` includes the student, a family size of 0 (nobody, not
even the student) or 1 (only the student, although a guardian is recorded)
is not plausible. These values are counted by parental status.

In [31]:
implausible_family = student_data['famsize'].isin([0, 1])

implausible_by_status = pd.crosstab(
    student_data.loc[implausible_family, 'famsize'],
    student_data.loc[implausible_family, 'Pstatus'].fillna('Missing')
)

implausible_by_status

Pstatus,A,Missing,T
famsize,,,
0.0,13,3,41
1.0,15,2,31


### Treatment of implausible family sizes

Each family size of 0 or 1 is replaced with the same rule as the missing
values. Seven of these students have an "other" guardian and receive the
"other" guardian median (7). The others receive the median of their
parental-status group (5 for parents apart, 6 for parents together).

Five students have no recorded parental status and none of them has an
"other" guardian, so no group from the rule is available for them. Their
family size is replaced with the median of all plausible family sizes (6).

In [32]:
overall_plausible_median = student_data.loc[plausible_family, 'famsize'].median()

family_replacement = family_reference.fillna(overall_plausible_median)

student_data_clean['famsize'] = student_data_clean['famsize'].mask(
    implausible_family, family_replacement
)

### Validation of implausible family-size replacement

The replacement values are counted by parental status and guardian type.
The checks confirm that every family size is now at least 2 and that
plausible recorded sizes and the previously filled missing values are
unchanged.

In [33]:
replacement_group = (
    student_data['Pstatus'].fillna('Missing')
    + ' / '
    + student_data['guardian'].map(
        lambda guardian: 'other' if guardian == 'other' else 'mother, father or missing'
    )
).rename('Parental status / guardian')

replaced_family = pd.crosstab(
    replacement_group[implausible_family],
    student_data_clean.loc[implausible_family, 'famsize']
)

assert student_data_clean['famsize'].ge(2).all()
assert student_data_clean.loc[plausible_family, 'famsize'].equals(
    student_data.loc[plausible_family, 'famsize']
)
assert student_data_clean.loc[
    student_data['famsize'].isna(), 'famsize'
].equals(family_reference[student_data['famsize'].isna()])

replaced_family

famsize,5.0,6.0,7.0
Parental status / guardian,,,
"A / mother, father or missing",24,0,0
A / other,0,0,4
"Missing / mother, father or missing",0,5,0
"T / mother, father or missing",0,69,0
T / other,0,0,3


### Treatment of missing school

One student has no recorded school. The value is replaced with the most
frequent school (the mode). With a single missing value, this has no
meaningful effect on the distribution. `.mode()` returns a Series because
several values can tie, so `[0]` selects the first.

In [34]:
school_mode = student_data_clean['school'].mode()[0]

student_data_clean['school'] = student_data_clean['school'].fillna(school_mode)

### Validation of school replacement

School counts before and after filling are compared.

In [35]:
school_check = pd.DataFrame({
    'Before': student_data['school'].value_counts(dropna=False),
    'After': student_data_clean['school'].value_counts(dropna=False)
}).fillna(0).astype(int)

assert student_data_clean['school'].isna().sum() == 0

school_check

,Before,After
school,,
GP,422,423
MS,226,226
NaN,1,0


### Remaining missing values

After all treatments, no missing values should remain in the working
dataset.

In [36]:
assert student_data_clean.isna().sum().sum() == 0

student_data_clean.shape

(649, 33)

### Working dataset

The prepared table is available as `student_data_clean` in this notebook.
Missing failures and absences are interpreted as zero by user decision.
The source meaning of those blanks remains unconfirmed. Missing Yes/No
responses are recorded as "Unknown". Missing study times are replaced
with the median of the student's school, and missing travel times with
the median of the student's school and residence type. Missing ratings
are replaced with the median of each feature. Missing parental status and
sex are recorded as "U"; missing parental education, parental occupation,
reason, and residence are recorded as "Unknown". Missing guardians and the
missing school are replaced with the mode. Missing and implausible (0 or 1)
family sizes are replaced with the median plausible family size of students
with an "other" guardian when the guardian is "other", otherwise of students
with the same parental status, and otherwise with the median of all
plausible family sizes. Further treatments require an explicit decision.
Duplicate records are evaluated above.

### Save the prepared dataset

The processed-data folder is created if needed, then the prepared dataset
is saved with its source row identifier.

In [37]:
output_path = Path('../data/processed/student_data_preprocessed.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

student_data_clean.to_csv(
    output_path,
    index=True,
    index_label='student_row_id'
)

print(output_path.resolve())

/Users/yassir/Data-Mining-Project/data/processed/student_data_preprocessed.csv
